# MySQL

Leemos el archivo csv y lo almacenamos en un dataframe de pandas

In [9]:
import pandas as pd

df = pd.read_csv('spotify-clean.csv')
df.sample(5)

,Unnamed: 0.1,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms
409,409,409,45SbQo5bq8f0hNdB70IyFC,Boyce Avenue,Save Your Tears,Save Your Tears,57,216800
24653,26104,26104,03sKaMN0mq5p3ok6z2xYVQ,Shin Fujishima;Tokyo Philharmonic Chorus;Natur...,My First Disney Original Soundtrack Best Japan...,When You Wish Upon A Star,31,151000
26355,27852,27852,3A9Qx8altSLoLZ8TNYcJwM,Delta Heavy,Paradise Lost,Ghost,47,255240
20514,21722,21722,3ZIHR8fa4ELkXo6fxwJ2St,Masicka,Rich,Rich,27,168946
32315,35720,35720,4cynZfPyLYlbSIy3HyStIO,Biu do Piseiro,O Véi Chegou!!,Aquece no Cacete,39,141672


Antes de conectarnos a nuestra BBDD de MySQL, instalaremos en nuestro environment: `pip install mysql-connector-python`.

Ahora, nos conectaremos a nuestra BBDD 'pipe' de MySQL y verificamos qué databases están creadas

In [10]:
import mysql.connector

mydb = mysql.connector.connect(
  host="localhost",
  user="root",
  password="changeme",
  database="pipe"
)

mycursor = mydb.cursor(buffered=True)

mycursor.execute("SHOW TABLES")

for x in mycursor:
  print(x)


('track',)


A continuación nos conectaremos a la BBDD 'pipe' y crearemos en ella la tabla 'spotify_tracks'

In [11]:
mycursor.execute("CREATE TABLE IF NOT EXISTS track (" \
"track_id varchar(250), " \
"track_name varchar(250), " \
"artists varchar(1000)," \
"album_name varchar(5120)," \
"popularity int," \
"duration_ms bigint," \
"PRIMARY KEY (track_id)" \
");")

#Show table structure
mycursor.execute("DESCRIBE track")

for x in mycursor:
  print(x)

('track_id', 'varchar(250)', 'NO', 'PRI', None, '')
('track_name', 'varchar(250)', 'YES', '', None, '')
('artists', 'varchar(1000)', 'YES', '', None, '')
('album_name', 'varchar(5120)', 'YES', '', None, '')
('popularity', 'int', 'YES', '', None, '')
('duration_ms', 'bigint', 'YES', '', None, '')


Comprobamos que la creación de la tabla `tracks` fue correcta. Procedemos a la inserción de los datos de nuestro dataset en la BBDD de MySQL.

Para ello, crearemos un bucle que nos inserte cada dato de nuestro dataframe en la tabla de mysql

In [12]:
for i in range(len(df)):
    sql = "INSERT INTO track (track_id, track_name, artists, album_name, popularity, duration_ms) VALUES (%s, %s, %s, %s, %s, %s)"
    val = (
        df.iloc[i]['track_id'],
        df.iloc[i]['track_name'],
        df.iloc[i]['artists'],
        df.iloc[i]['album_name'],
        int(df.iloc[i]['popularity']),    # convertir a tipo int nativo
        int(df.iloc[i]['duration_ms'])    # convertir a tipo int nativo
    )
    mycursor.execute(sql, val)
    
mydb.commit()

IntegrityError: 1062 (23000): Duplicate entry '5SuOikwiRyPMVoIQDJUgSV' for key 'track.PRIMARY'